In [ ]:
#!/usr/bin/env python3
"""
NB11: Fairness & Equity Analysis
Evaluates the geographical equality of AED coverage across provinces
using standard econ metrics: Lorenz Curves, Gini Coefficient, and Theil Index.
"""



# NB 11 — Fairness & Equity Analysis

Assesses the socio-geographic distribution of AED coverage.
Are rural provinces systematically underserved compared to urban ones?
We use standard inequality metrics (Gini Coefficient, Lorenz Curve, Theil Index)
on the proportion of covered missions per province.

**Expected runtime: < 1 minute**


In [ ]:
Setup
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.')
V3_DIR   = PROJECT_ROOT / 'mda_project' / 'data' / 'processed_v3'
OUT_DIR  = PROJECT_ROOT / 'mda_project' / 'data' / 'output'
FIG_DIR  = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)




In [ ]:
Load Data
mission = pd.read_parquet(V3_DIR / 'mission_records_v3.parquet')
m = mission.dropna(subset=['province', 'dist_to_aed_km']).copy()

# Group by province
prov_stats = m.groupby('province').agg(
    n_missions=('mission_id', 'count'),
    covered=('dist_to_aed_km', lambda x: (x <= 0.5).sum())
).reset_index()

# Filter low-data provinces (OVL, VBR, WVL) which have <10 records due to geocoding issues
prov_stats = prov_stats[prov_stats['n_missions'] > 100].copy()

# Coverage rate per province
prov_stats['coverage_rate'] = prov_stats['covered'] / prov_stats['n_missions']

# Sort for Lorenz Curve (sort by coverage rate)
prov_stats = prov_stats.sort_values('coverage_rate')

prov_stats['cum_missions'] = prov_stats['n_missions'].cumsum() / prov_stats['n_missions'].sum()
prov_stats['cum_covered'] = prov_stats['covered'].cumsum() / prov_stats['covered'].sum()

print("Province coverage stats:")
print(prov_stats[['province', 'n_missions', 'covered', 'coverage_rate']].to_string())




In [ ]:
Gini & Theil
# Gini formula for grouped data
def calc_gini(x, weights):
    # x: coverage values, weights: population/missions
    idx = np.argsort(x)
    x = np.asarray(x)[idx]
    w = np.asarray(weights)[idx]
    w_cum = np.cumsum(w) / np.sum(w)
    
    # Area under Lorenz curve
    area = np.sum((w_cum[1:] - w_cum[:-1]) * (x[1:] + x[:-1]) / 2)
    gini = 1 - 2 * area # This is a simplified proxy since perfect equality is the diagonal
    
    # Alternative: standard numeric Gini on coverage rates
    # The Gini coefficient is the ratio of half the relative mean absolute difference to the mean
    n = len(x)
    numerator = 2 * np.sum((np.arange(1, n+1) * x))
    denominator = n * np.sum(x)
    gini = (numerator / denominator) - (n + 1) / n
    return float(gini)

def calc_theil(x, weights):
    # Theil T
    x = np.asarray(x)
    w = np.asarray(weights)
    mu = np.average(x, weights=w)
    # T = sum( w_i/W * (x_i/mu) * ln(x_i/mu) )
    ratio = x / mu
    ratio[ratio <= 0] = 1e-10 # numeric stab
    return np.average(ratio * np.log(ratio), weights=w)

gini = calc_gini(prov_stats['coverage_rate'], prov_stats['n_missions'])
theil = calc_theil(prov_stats['coverage_rate'], prov_stats['n_missions'])

print(f"\nBaseline Gini Coefficient: {gini:.4f} (0 = perfect equality)")
print(f"Baseline Theil Index: {theil:.4f} (0 = perfect equality)")

# Export table
prov_stats.to_csv(FIG_DIR / 'table_11_equity_stats.csv', index=False)




In [ ]:
Figure: Lorenz Curve
print("Rendering Fig 8 (Equity/Lorenz)...")
fig8, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), dpi=200)

# (a) Lorenz Curve
# Prepend 0,0
x_lz = np.concatenate([[0], prov_stats['cum_missions'].values])
y_lz = np.concatenate([[0], prov_stats['cum_covered'].values])

ax1.plot([0, 1], [0, 1], color='#333', ls='--', label='Line of Perfect Equality')
ax1.plot(x_lz, y_lz, color='#c0392b', lw=2.5, marker='o', label=f'Lorenz Curve (Gini = {gini:.3f})')
ax1.fill_between(x_lz, x_lz, y_lz, color='#c0392b', alpha=0.1)

ax1.set_title('(g) Lorenz Curve of AED Coverage Equality', fontweight='bold')
ax1.set_xlabel('Cumulative Proportion of Emergency Missions')
ax1.set_ylabel('Cumulative Proportion of Covered Missions')
ax1.legend()
ax1.grid(ls=':', alpha=0.6)

# (b) Bar chart
bars = ax2.bar(prov_stats['province'], prov_stats['coverage_rate']*100, color='#2980b9', edgecolor='white')

mean_cov = prov_stats['covered'].sum() / prov_stats['n_missions'].sum() * 100
ax2.axhline(mean_cov, color='#e74c3c', ls='--', label=f'National Mean ({mean_cov:.1f}%)')

# Annotate values
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=10)

ax2.set_title('(h) 500m Coverage Rate by Province', fontweight='bold')
ax2.set_ylabel('Coverage Rate [%]')
ax2.legend()
ax2.set_ylim(0, 100)
ax2.grid(axis='y', ls=':', alpha=0.6)

fig8.tight_layout()
fig8.savefig(FIG_DIR / 'fig8_equity_lorenz.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.close(fig8)
print("  -> saved fig8_equity_lorenz.png")
print("\n=== NB11 Complete ===")

